# 04b 子样本稳定性：按年份段检查 `stem=丙`

目的：看 `stem=丙` 的收益差异是否集中在某一段，还是跨期更稳定。

分段：`2010-2015`、`2016-2020`、`2021-now`

输入：
- `data/clean/market_ganzhi_{ts_code}.csv.gz`

输出（写入 `data/clean/robustness/`）：
- `subsample_bing.csv`


In [1]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from patsy import build_design_matrices
from scipy import stats

warnings.filterwarnings('ignore')

# statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

# plots
import matplotlib.pyplot as plt
import seaborn as sns


def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()

    for candidate in [here] + list(here.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    for candidate in here.glob('*'):
        if candidate.is_dir() and (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    return here


ROOT = find_project_root()
print('PROJECT_ROOT =', ROOT)

CLEAN_DIR = ROOT / 'data/clean'
ROBUST_DIR = CLEAN_DIR / 'robustness'
ROBUST_DIR.mkdir(parents=True, exist_ok=True)

# =====================
# Config (edit as needed)
# =====================
TS_CODES = ['000300.SH', '000852.SH', '000001.SH', '399001.SZ']

# Control-variable models to run
GROUP_COLS = ['stem', 'branch']
RUN_GANZHI_DAY_MODELS = True

# Permutation test (raw series; global)
RANDOM_SEED = 20260213
N_PERM = 1000

# Permutation on controls-only residuals (more conservative)
N_PERM_RESID = 2000

# HAC / Newey-West (time-series dependence)
HAC_MAXLAGS = 5
HAC_MAXLAGS_SWEEP = [0, 1, 3, 5, 10]

# Block bootstrap (controls-only residuals)
N_BOOT = 1000
BOOT_BLOCK_LEN = 10

# Walk-forward
TRAIN_YEARS = 5
TRAIN_YEARS_SWEEP = [3, 5, 7]

STEMS_ORDER = list('甲乙丙丁戊己庚辛壬癸')
BRANCHES_ORDER = list('子丑寅卯辰巳午未申酉戌亥')

# Phase 2: Li-Chun year element interaction (day_group × year_element)
RUN_PHASE2_INTERACTIONS = True
PHASE2_DAY_GROUPS = ['stem', 'branch', 'ganzhi_day']
YEAR_ELEMENT_ORDER = ['木', '火', '土', '金', '水']
PHASE2_P_THRESHOLD = 0.10
PHASE2_GATE_Q = 0.10
PHASE2_GATE_MIN_SHARE = 0.75


def fdr_bh(pvals: np.ndarray) -> np.ndarray:
    # Benjamini-Hochberg FDR q-values for a 1D array.
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    q = np.empty(n, dtype=float)
    prev = 1.0

    for rank in range(n - 1, -1, -1):
        i = order[rank]
        val = p[i] * n / (rank + 1)
        prev = min(prev, val)
        q[i] = prev

    return np.clip(q, 0.0, 1.0)


def load_market_ganzhi(ts_code: str) -> pd.DataFrame:
    path = CLEAN_DIR / f'market_ganzhi_{ts_code}.csv.gz'
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}. Run notebooks 01-03 first.')
    df = pd.read_csv(path, compression='gzip', dtype={'trade_date': str})
    if df.empty:
        raise ValueError(f'Empty data: {path}')
    return df


def add_calendar_controls(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out['trade_date'], format='%Y%m%d')
    out['date'] = dt
    out['weekday'] = dt.dt.weekday.astype(int)
    out['month'] = dt.dt.month.astype(int)
    out['year'] = dt.dt.year.astype(int)
    return out


def set_category_orders(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'stem' in out.columns:
        out['stem'] = pd.Categorical(out['stem'], categories=STEMS_ORDER, ordered=True)
    if 'branch' in out.columns:
        out['branch'] = pd.Categorical(out['branch'], categories=BRANCHES_ORDER, ordered=True)
    return out




PROJECT_ROOT = D:\Work\中国传统投资\风水五行阴阳天干地支


## 1) 子样本分段（Welch t-test）


In [2]:
def year_bucket(y: int) -> str:
    if y <= 2015:
        return '2010-2015'
    if y <= 2020:
        return '2016-2020'
    return '2021-now'


def subsample_bing(ts_code: str) -> pd.DataFrame:
    df = load_market_ganzhi(ts_code)
    df = add_calendar_controls(df)
    df = df.dropna(subset=['ret_1d'])
    df['bucket'] = df['year'].map(year_bucket)

    records = []
    for b, sub in df.groupby('bucket'):
        all_mean = float(sub['ret_1d'].mean())
        bing = sub[sub['stem'] == '丙']
        other = sub[sub['stem'] != '丙']

        if len(bing) < 20 or len(other) < 20:
            pval = np.nan
        else:
            t = sm.stats.ttest_ind(bing['ret_1d'], other['ret_1d'], usevar='unequal')[1]
            pval = float(t)

        records.append(
            {
                'ts_code': ts_code,
                'bucket': b,
                'n_all': int(len(sub)),
                'n_bing': int(len(bing)),
                'mean_ret_all': all_mean,
                'mean_ret_bing': float(bing['ret_1d'].mean()) if len(bing) else np.nan,
                'effect_bing_minus_all': float(bing['ret_1d'].mean()) - all_mean if len(bing) else np.nan,
                'p_value_welch': pval,
            }
        )

    return pd.DataFrame.from_records(records).sort_values(['ts_code', 'bucket'])


subs = pd.concat([subsample_bing(c) for c in TS_CODES], ignore_index=True)
out_path = ROBUST_DIR / 'subsample_bing.csv'
subs.to_csv(out_path, index=False)
print('saved:', out_path)
subs




saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\robustness\subsample_bing.csv


,ts_code,bucket,n_all,n_bing,mean_ret_all,mean_ret_bing,effect_bing_minus_all,p_value_welch
0,000300.SH,2010-2015,1456,145,0.000159,-0.002137,-0.002296,0.084168
1,000300.SH,2016-2020,1218,125,0.000353,-0.000397,-0.000750,0.495981
2,000300.SH,2021-now,1242,125,-0.000026,-0.002870,-0.002844,0.012249
3,000852.SH,2010-2015,1456,145,0.000785,-0.000159,-0.000943,0.544316
4,000852.SH,2016-2020,1218,125,-0.000249,-0.003118,-0.002869,0.044728
5,000852.SH,2021-now,1242,125,0.000283,-0.001809,-0.002092,0.168941
6,000001.SH,2010-2015,1456,145,0.000163,-0.001687,-0.001850,0.134826
7,000001.SH,2016-2020,1218,125,0.000055,-0.001037,-0.001092,0.308133
8,000001.SH,2021-now,1242,125,0.000179,-0.001719,-0.001898,0.065016
9,399001.SZ,2010-2015,1456,145,0.000095,-0.001848,-0.001943,0.175894
